# CNN vs SVM: the ablation moves the two models in opposite directions

Fifteen SVM checkpoints (3 variants × 5 splits), fitted on the CNN's own partitions so
every SVM checkpoint is paired with a CNN checkpoint.

**The starting hypothesis was wrong.** The old SVM funnels out-of-domain predictions into
`C_diminished_4` — 37% of `thinkpad-2` against a true rate of 2.8% — and that class also
holds the 16 digital-silence clips, which are exactly `0.0` in all 40,608 dimensions. The
obvious inference was that those clips caused the sink and that removing them would
dissolve it.

They do not. Removing them removes the training-set outlier (`train_l2_max` 1235 → 621)
and makes out-of-domain transfer *worse*, on every seed.

**What is actually happening.** Those 16 all-zero rows were inflating the scaler's
per-dimension variance by up to 2.26×. Dropping them tightens the scaler, so scaled norms
grow — about 4% in domain but 11% out of domain. Kernel values fall. More rows land
outside the kernel's support. As K→0 each one-vs-one decision reduces to its intercept,
the vote vector becomes constant, and one class absorbs everything.

That class is `C_diminished_4` on all 15 checkpoints — but always by a margin of **exactly
one vote**, and it matches a prediction computable from `intercept_` signs alone before any
data is seen. Its identity is near-arbitrary; only the collapse is structural. The silent
clips being `C_diminished_4` was a coincidence that made a wrong hypothesis look plausible.

The CNN does not collapse this way: its most-predicted class on `thinkpad-2` takes
5.3–10.4% and is a *different* class on each of its ten checkpoints. Its first layer is
`BatchNormalization(axis=1)`, which renormalizes per frequency bin at inference, so a
global gain or tilt shift does not move it off-manifold the way a fixed `StandardScaler`
plus a global distance metric does.

Two further results the tables below carry:

- **The SVM is not uniformly worse.** It ties the CNN in domain and beats it on `flow` and
  `vivo`. The deficit is concentrated on `thinkpad` and the keyboard shift.
- **`C = 1000` is inactive.** `dual_coef_abs_max ≈ 1.4` against the box constraint and
  `bounded_fraction = 0.0` on every checkpoint, so `C = 10` gives an identical model. The
  published grid reported `C = 1000` as "best" over a region where `C` does nothing. Any
  retune must vary gamma only.

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd

HERE = Path("/home/seya/code/chord-detection/training/notebooks/svm-latest")
sys.path.insert(0, str(HERE))
import svm_common as C

pd.set_option("display.width", 160)

svm_ood = pd.read_csv(C.OUT_DIR / "ood_seeds.csv").assign(model="SVM")
cnn_ood = pd.read_csv(C.CNN_DIR / "results" / "ood_seeds.csv").assign(model="CNN")
ood = pd.concat([cnn_ood, svm_ood], ignore_index=True)

print("SVM checkpoints:", svm_ood.groupby("variant").seed.nunique().to_dict())
ood.pivot_table(index=["variant", "dataset"], columns="model",
                values="accuracy", aggfunc="mean").round(4)

SVM checkpoints: {'clean': 5, 'clean-fixed': 5, 'orig': 5}


model                      CNN     SVM
variant     dataset                   
clean       flow        0.9385  0.9814
            flow-2      0.8742  0.6804
            thinkpad    0.9839  0.7147
            thinkpad-2  0.8649  0.4185
            vivo        0.8765  0.9724
clean-fixed flow           NaN  0.9772
            flow-2         NaN  0.7010
            thinkpad       NaN  0.7172
            thinkpad-2     NaN  0.4353
            vivo           NaN  0.9649
orig        flow        0.9533  0.9926
            flow-2      0.8124  0.7575
            thinkpad    0.9751  0.8251
            thinkpad-2  0.7988  0.5207
            vivo        0.9153  0.9756

In [2]:
# The headline: the same 28-clip removal moves the two models in opposite directions.
sm = svm_ood.groupby("variant")["accuracy"].mean()
cm = cnn_ood.groupby("variant")["accuracy"].mean()
print("mean accuracy over the 5 recorded sets, 5 seeds each\n")
print(f"{'':16s}{'orig':>8}{'clean':>9}{'delta':>11}")
print(f"{'CNN':16s}{cm['orig']:8.4f}{cm['clean']:9.4f}{(cm['clean']-cm['orig'])*100:+9.1f} pts")
print(f"{'SVM':16s}{sm['orig']:8.4f}{sm['clean']:9.4f}{(sm['clean']-sm['orig'])*100:+9.1f} pts")
print(f"\n{'SVM clean-fixed':16s}{'':8s}{sm['clean-fixed']:9.4f}{(sm['clean-fixed']-sm['orig'])*100:+9.1f} pts")
print("\ndecomposition of the SVM drop")
print(f"  data removal  (clean-fixed - orig)   {(sm['clean-fixed']-sm['orig'])*100:+.2f} pts")
print(f"  split redraw  (clean - clean-fixed)  {(sm['clean']-sm['clean-fixed'])*100:+.2f} pts")

# in-domain, for contrast: both models saturate and cannot be told apart
ks = pd.read_csv(C.OUT_DIR / "kfold.csv").groupby("variant")["test_accuracy"].mean()
kc = pd.read_csv(C.CNN_DIR / "results" / "kfold.csv").groupby("variant")["test_accuracy"].mean()
print("\nin-domain 5-fold, same partitions (a memorization check, not a comparison)")
print(f"  CNN  orig {kc['orig']:.4f}  clean {kc['clean']:.4f}")
print(f"  SVM  orig {ks['orig']:.4f}  clean {ks['clean']:.4f}")

mean accuracy over the 5 recorded sets, 5 seeds each

                    orig    clean      delta
CNN               0.8910   0.9076     +1.7 pts
SVM               0.8143   0.7535     -6.1 pts

SVM clean-fixed            0.7591     -5.5 pts

decomposition of the SVM drop
  data removal  (clean-fixed - orig)   -5.52 pts
  split redraw  (clean - clean-fixed)  -0.56 pts

in-domain 5-fold, same partitions (a memorization check, not a comparison)
  CNN  orig 0.9999  clean 0.9999
  SVM  orig 1.0000  clean 1.0000


## The ablation, and what `clean-fixed` separates

`sink_share_excess` is the fraction of predictions absorbed by the single most-predicted
class, minus the 1/36 a balanced set gives anyway.

`clean` differs from `orig` in two things at once — the 28 clips *and* a redrawn split.
`clean-fixed` keeps `orig`'s partition and removes only the clips, so `clean-fixed − orig`
is the effect of the data alone and `clean − clean-fixed` is the effect of the redraw.
This is the strictly-paired ablation `sn-article-draft-8.tex:223` notes is missing on the
CNN side.

In [3]:
sink = pd.read_csv(C.OUT_DIR / "sink_seeds.csv")

# SVM: sink share per variant/dataset, averaged over seeds
svm_sink = sink.pivot_table(index="dataset", columns="variant",
                            values="sink_share_excess", aggfunc="mean") * 100

# CNN control: same statistic, recomputed from its confusion matrices
rows = []
for f in sorted((C.CNN_DIR / "results" / "confusion").glob("*.csv")):
    ds, variant, seed = f.stem.rsplit("_", 2)
    col = pd.read_csv(f, index_col=0).sum(0)
    rows.append({"dataset": ds, "variant": variant, "seed": seed,
                 "sink_class": col.idxmax(),
                 "sink_share_excess": (col.max() / col.sum() - 1 / 36) * 100})
cnn_sink = pd.DataFrame(rows)

out = svm_sink.copy()
out.columns = [f"SVM {c}" for c in out.columns]
out["CNN orig"] = cnn_sink[cnn_sink.variant == "orig"].groupby("dataset")["sink_share_excess"].mean()
out["CNN clean"] = cnn_sink[cnn_sink.variant == "clean"].groupby("dataset")["sink_share_excess"].mean()
print("Excess share of predictions absorbed by the single most-predicted class "
      "(percentage points above 1/36)")
display(out.round(1))

print("\nWhich class is the sink, SVM:")
display(sink.pivot_table(index="dataset", columns="variant", values="sink_class",
                         aggfunc=lambda s: s.value_counts().index[0]))
print("\nDistinct sink classes across CNN checkpoints (a real attractor would be 1):")
display(cnn_sink.groupby(["variant", "dataset"])["sink_class"].nunique().unstack())

Excess share of predictions absorbed by the single most-predicted class (percentage points above 1/36)


,SVM clean,SVM clean-fixed,SVM orig,CNN orig,CNN clean
dataset,,,,,
flow,0.9,1.1,0.6,1.7,3.1
flow-2,20.9,19.7,17.4,4.2,3.7
test,0.0,0.0,0.0,NaN,NaN
thinkpad,22.1,20.7,15.0,0.9,1.1
thinkpad-2,47.7,45.2,39.9,4.5,4.3
val,0.0,0.0,0.0,NaN,NaN
vivo,1.5,1.8,1.4,2.2,3.0



Which class is the sink, SVM:


variant,clean,clean-fixed,orig
dataset,,,
flow,C_diminished_4,C_diminished_4,C_diminished_4
flow-2,C_diminished_4,C_diminished_4,C_diminished_4
test,A#_diminished_4,A#_diminished_4,A#_diminished_4
thinkpad,C_diminished_4,C_diminished_4,C_diminished_4
thinkpad-2,C_diminished_4,C_diminished_4,C_diminished_4
val,A#_diminished_4,A#_diminished_4,A#_diminished_4
vivo,G_minor_4,G_minor_4,G_minor_4



Distinct sink classes across CNN checkpoints (a real attractor would be 1):


dataset,flow,flow-2,thinkpad,thinkpad-2,vivo
variant,,,,,
clean,5,3,5,4,4
orig,5,5,5,5,5


In [ ]:
# How much of the accuracy loss does the sink account for?
# Loss is measured against 1.000, the in-domain ceiling both models reach.
m = sink.merge(pd.read_csv(C.OUT_DIR / "ood_seeds.csv"),
               on=["variant", "seed", "dataset"], how="inner")
m["acc_lost_pts"] = (1.0 - m["accuracy"]) * 100
m["sink_excess_pts"] = m["sink_share_excess"] * 100
m["attributed"] = m["sink_excess_pts"] / m["acc_lost_pts"].replace(0, np.nan)

att = m.groupby(["variant", "dataset"])[
    ["acc_lost_pts", "sink_excess_pts", "attributed"]].mean()
display(att.round(3))

r = m[["sink_excess_pts", "acc_lost_pts"]].corr().iloc[0, 1]
print(f"correlation between sink excess and accuracy lost, across all cells: r = {r:.3f}")

## Recall by quality — read this one carefully

These are *collapsed* recalls: a `C_major` predicted as `G_major` counts as a quality hit
and a root miss. That separates "wrong chord type" from "wrong root", but it means the
SVM's diminished recall of ~0.99 on `thinkpad-2` is **not** evidence that it handles
diminished chords well. The sink class is itself a diminished chord, so every clip swept
into it scores as quality-correct if its true label was any diminished chord, and
quality-wrong otherwise. Diminished ~0.99 against major ~0.33 is a direct readout of the
collapse, not a property of the chord type.

The CNN's ordering is the honest one: diminished is its *worst* quality under shift
(0.745–0.802 on `thinkpad-2`), which is what a model that is actually discriminating
should show — diminished triads are the closest-spaced of the three qualities.

An earlier reading of this table as a "quality inversion between the two models" was
measuring the sink, not the models.

In [4]:
# Quality inversion: does the sink still prop diminished up once the clips are gone?
qr = pd.read_csv(C.OUT_DIR / "quality_root_recall_seeds.csv")
svm_q = (qr[qr.axis == "quality"]
         .pivot_table(index=["variant", "dataset"], columns="group", values="recall"))

cnn_rec = pd.read_csv(C.CNN_DIR / "results" / "per_class_recall_seeds.csv")
cnn_rec["group"] = cnn_rec["class"].str.split("_").str[1]
cnn_q = cnn_rec.pivot_table(index=["variant", "dataset"], columns="group", values="recall")

print("SVM recall by chord quality")
display(svm_q.round(3))
print("\nCNN recall by chord quality (diminished is its *worst* quality under shift)")
display(cnn_q.round(3))

SVM recall by chord quality


group                   diminished  major  minor
variant     dataset                             
clean       flow             0.999  0.984  0.962
            flow-2           0.977  0.662  0.693
            test             1.000  1.000  1.000
            thinkpad         0.990  0.678  0.597
            thinkpad-2       0.985  0.330  0.355
            val              1.000  1.000  1.000
            vivo             0.925  1.000  0.996
clean-fixed flow             1.000  0.984  0.949
            flow-2           0.988  0.662  0.706
            test             1.000  1.000  1.000
            thinkpad         0.988  0.679  0.584
            thinkpad-2       0.988  0.339  0.372
            val              1.000  1.000  1.000
            vivo             0.908  0.996  0.996
orig        flow             1.000  0.990  0.989
            flow-2           1.000  0.712  0.764
            test             1.000  1.000  1.000
            thinkpad         1.000  0.774  0.737
            thinkpad-2       0.998  0.428  0.476
            val              1.000  1.000  1.000
            vivo             0.933  1.000  0.998


CNN recall by chord quality (diminished is its *worst* quality under shift)


group               diminished  major  minor
variant dataset                             
clean   flow             0.924  0.937  0.954
        flow-2           0.828  0.890  0.905
        thinkpad         0.989  0.980  0.983
        thinkpad-2       0.802  0.888  0.905
        vivo             0.812  0.910  0.907
orig    flow             0.934  0.980  0.947
        flow-2           0.768  0.838  0.831
        thinkpad         0.964  0.986  0.975
        thinkpad-2       0.745  0.844  0.808
        vivo             0.865  0.978  0.902

In [5]:
# Mechanism diagnostics: does the sink match the closed-form K->0 prediction,
# and do the cleaned variants pull the scaled norms and support counts back?
diag = pd.read_csv(C.OUT_DIR / "diagnostics.csv")
cols = ["n_sv_total", "sv_fraction", "dual_coef_abs_max", "bounded_fraction",
        "train_l2_median", "train_l2_max", "sink_class_zero_kernel",
        "fast_predict_validated", "fast_predict_min_abs_dec", "predict_ms_per_clip"]
display(diag.groupby("variant")[["n_sv_total", "sv_fraction", "dual_coef_abs_max",
                                 "bounded_fraction", "train_l2_median", "train_l2_max",
                                 "predict_ms_per_clip"]].mean().round(4))
print("\nclosed-form K->0 sink class per checkpoint (predicted before seeing any data):")
display(diag.groupby(["variant", "sink_class_zero_kernel"]).size())
print("\nempirical sink matches that prediction:")
display(sink.groupby(["variant", "dataset"])["sink_matches_zero_kernel_class"].mean().unstack())
assert diag["fast_predict_validated"].all(), "a fast-predict gate failed"
print("\nall fast-predict gates passed; min |decision| seen:",
      f"{diag['fast_predict_min_abs_dec'].min():.3e}")

,n_sv_total,sv_fraction,dual_coef_abs_max,bounded_fraction,train_l2_median,train_l2_max,predict_ms_per_clip
variant,,,,,,,
clean,4027.8,0.7022,1.3419,0.0,197.8853,621.8891,1.8818
clean-fixed,4004.4,0.6979,1.2778,0.0,197.8461,621.4447,1.5471
orig,3892.2,0.6757,1.3339,0.0,189.6451,1234.9198,1.5888



closed-form K->0 sink class per checkpoint (predicted before seeing any data):


variant      sink_class_zero_kernel
clean        C_diminished_4            5
clean-fixed  C_diminished_4            5
orig         C_diminished_4            5
dtype: int64


empirical sink matches that prediction:


dataset,flow,flow-2,test,thinkpad,thinkpad-2,val,vivo
variant,,,,,,,
clean,0.8,1.0,0.0,1.0,1.0,0.0,0.0
clean-fixed,0.6,1.0,0.0,1.0,1.0,0.0,0.0
orig,1.0,1.0,0.0,1.0,1.0,0.0,0.0



all fast-predict gates passed; min |decision| seen: 1.770e-07
